## Exploiting a Classification System  
This script is based on the HopSkipJump attack as provided in the Adversarial Robustness Toolbox attack.  It implements the HopSkipJump attack developed by Chen et al using a family of algorithms based on _a novel estimate of the gradient direction using binary information at the decision boundary_. This script uses the approach to overlay a target classification image on a victim image in such a way as to achieve the AI classification desired, and then to iteratively improve the quality of the compromised image.

Let's start by ignoring warnings to the extent we can.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

Let's start by downloading the imagenet stubs from github.  This provides us with some image functions that we can use, and a set of demonstration images.

In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/nottombrown/imagenet_stubs
sys.path.append("..")

We can now import the libraries we need, including some from the adversarial robustness toolbox.

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals
%matplotlib inline

import tensorflow as tf
tf.compat.v1.disable_eager_execution()

import imagenet_stubs
import numpy as np
import tensorflow.keras
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.models import Model
from matplotlib import pyplot as plt

from art.estimators.classification import KerasClassifier
from art.attacks.evasion import HopSkipJump
from art.utils import to_categorical

## Model Definition  
We can now define the model with shape 224x224x3 (ie, a square 224 pixel image stored as rgb).  We then initialise the image pixels.  We define a ResNet50 model. ResNet50 is a deep learning model for image classification that was introduced by Microsoft researchers in 2015. It is a deep convolutional neural network that can classify images into 1,000 categories, including common objects, animals, and scenes.  We then create an instance of the AI classifier using the ResNet model.
  
The classifications are listed at https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/. 

In [ ]:
mean_imagenet = np.zeros([224, 224, 3])
mean_imagenet[...,0].fill(103.939)
mean_imagenet[...,1].fill(116.779)
mean_imagenet[...,2].fill(123.68)
model = ResNet50(weights='imagenet')
classifier = KerasClassifier(clip_values=(0, 255), model=model, preprocessing=(mean_imagenet, np.ones([224, 224, 3])))

## Load Images  
In this section we load two jpg files from the imagenet images directory, getting their full path by looping through the set of images returned by imagenet_stubs. One image is the target of our attack, i.e. the image which we will corrupt in order to have it wrongly classified. The second is the image representing the classification we want to be identified for the compromised target image. We load each of the images and store them in an array.


In [ ]:
target_image_name = 'gazelle.jpg'
class_image_name = 'tractor.jpg'

for image_path in imagenet_stubs.get_image_paths():
    if image_path.endswith(target_image_name):
        target_image = image.load_img(image_path, target_size=(224, 224))
        target_image = image.img_to_array(target_image)
    if image_path.endswith(class_image_name):
        class_image = image.load_img(image_path, target_size=(224, 224))
        class_image = image.img_to_array(class_image)


Now we'll output the images to show what they look like. We'll also use the classifier to identify their true classifications.

In [ ]:
print("Target image is currently: ", np.argmax(classifier.predict(np.array([target_image]))[0]))
plt.imshow(target_image.astype(np.uint))
plt.show()
print("Classification image is: ", np.argmax(classifier.predict(np.array([class_image]))[0]))
plt.imshow(class_image.astype(np.uint))
plt.show()

## Run the Attack  
We can see we have a target image which is a gazelle. In this exploit, we want to make the ResNet50 AI classifier classify the image as a tractor. We'll essentially overlay a form of the tractor image into the gazelle image, sufficient to make the classifier accept it as the adversarial classification.  

We'll set up an attack handle based on the ResNet50 classifier and the target image, and set its evaluation parameters. The attack will run for 10 steps.  We'll then set up a mask based on the shape of the images and set an acceptable level of probability.  We'll set _adversarial_image_ to the initial state of the target image.

In [ ]:
attack = HopSkipJump(classifier=classifier, targeted=True, max_iter=0, max_eval=1000, init_eval=10)
iter_step = 10
adversarial_image = np.array([class_image])
mask = np.random.binomial(n=1, p=0.9, size=np.prod(target_image.shape))
mask = mask.reshape(target_image.shape)

We're now ready to exploit the target image. We'll run the exploit 20 times, each time seeking to reduce the level of layer 2 error.  At each turn, we'll output the increasingly compromised adversarial image together with its classification and error level.

In [ ]:
for i in range(20):
    adversarial_image = attack.generate(x=np.array([target_image]), y=to_categorical([866], 1000), x_adv_init=adversarial_image, resume=True, mask=mask)
    attack.max_iter = iter_step
    
    print("Adversarial image at step %d." % (i * iter_step), "L2 error", 
          np.linalg.norm(np.reshape(adversarial_image[0] - target_image, [-1])),
          "and class label %d." % np.argmax(classifier.predict(adversarial_image)[0]))
    plt.imshow(adversarial_image[0].astype(np.uint))
    plt.show(block=False)
  


## Attack Complete  
We can see that we now have an image which looks like a gazelle, but is classified as a tractor with classification code 866.  At each iteration we reduce the layer 2 error, achieving a quite respectable compromised image after 200 steps.  We quickly got down to a level of error which was acceptable, and we can see the improvements reducing at each iteration.